In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time


In [10]:
start_url = 'https://www.musashino-u.ac.jp/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

visited = set()  # 訪問済みURL
url_titles = {}  # URL→タイトル

# 除外したい外部サイト
EXCLUDE_DOMAINS = [
    "facebook.com",
    "twitter.com",
    "instagram.com",
    "line.me",
    "youtube.com",
    "x.com",
    "linkedin.com"
]

In [11]:
def is_excluded(url):
    return any(domain in url for domain in EXCLUDE_DOMAINS)

def crawl(url):
    if url in visited or is_excluded(url):
        return
    visited.add(url)
    
    try:
        res = requests.get(url, headers=headers, timeout=5)
        res.encoding = res.apparent_encoding
        
        if res.status_code != 200:
            url_titles[url] = f"アクセスできません ({res.status_code})"
            return
        
        soup = BeautifulSoup(res.text, 'html.parser')
        title_tag = soup.find('title')
        url_titles[url] = title_tag.get_text().strip() if title_tag else "No Title"
        print(f"{url} → {url_titles[url]}")
        
        # 再帰的関数：同じ処理を何度も繰り返してリンクをたどる
        for a in soup.find_all('a', href=True):
            href = a['href']
            full_url = urljoin(start_url, href)
            
            # 武蔵野大学のドメインをクロールする
            if 'musashino-u.ac.jp' in full_url and not is_excluded(full_url):
                crawl(full_url)
        
        time.sleep(0.5)  # サーバーに負荷をかけないよう停止する
    except Exception as e:
        url_titles[url] = "アクセスできません"

In [12]:
# クロールを開始する
crawl(start_url)

https://www.musashino-u.ac.jp/ → 武蔵野大学
https://www.musashino-u.ac.jp/#main → 武蔵野大学
https://ef.musashino-u.ac.jp/donation/ → ご寄付のお願い | 学校法人武蔵野大学
https://ef.musashino-u.ac.jp/donation/ → ご寄付のお願い | 学校法人武蔵野大学
https://www.musashino-u.ac.jp/news/2025/?category=活動レポート → ニュース  | 武蔵野大学
https://www.musashino-u.ac.jp/access.html → 交通アクセス | 武蔵野大学
https://www.musashino-u.ac.jp/news/2025/?category=活動レポート → ニュース  | 武蔵野大学
https://www.musashino-u.ac.jp/access.html → 交通アクセス | 武蔵野大学
https://www.musashino-u.ac.jp/admission/request.html → 資料請求 | 入試情報 | 武蔵野大学
https://www.musashino-u.ac.jp/contact.html → お問い合わせ | 武蔵野大学
https://www.musashino-u.ac.jp/admission/request.html → 資料請求 | 入試情報 | 武蔵野大学
https://www.musashino-u.ac.jp/contact.html → お問い合わせ | 武蔵野大学
https://www.musashino-u.ac.jp/prospective-students.html → 武蔵野大学で学びたい方 | 武蔵野大学
https://www.musashino-u.ac.jp/students.html → 在学生の方 | 武蔵野大学
https://www.musashino-u.ac.jp/prospective-students.html → 武蔵野大学で学びたい方 | 武蔵野大学
https://www.musashino-u.ac.jp/students.html →